# DO NOT CHANGE HERE
change ldhmm_3_tr_hmm_groups_of_states_different_alleles.ipynb
and apply changes according to !CHANGE! tags

In [1]:
import numpy as np
import pandas as pd
import sys
PATH_TO_PREDICTOR_HOME = "../.."
sys.path.append(PATH_TO_PREDICTOR_HOME)
METHOD = "hmm_pomegranate"


# 1. Read data

In [2]:
DATA_PATH = 'D:/Google Drive/Work/ITMO/data/length_dependent_hmm/per_length_per_kfold_split/' # windows
# DATA_PATH = '/home/denklewer/Data/length_dependent_hmm/' #ubuntu
TARGET_PATH_TO_RESULTS = "experiment_results/alleles_with_groups_bwv/" #!CHANGE for bwv!

In [3]:
from data_reading_methods import get_available_alleles, read_data

ALLELES = get_available_alleles(DATA_PATH)

per_allele_per_kfold_per_length_binders_train = read_data(DATA_PATH, ALLELES, "train", "binders")
per_allele_per_kfold_per_length_binders_test = read_data(DATA_PATH, ALLELES, "test", "binders")
per_allele_per_kfold_per_length_non_binders_train = read_data(DATA_PATH, ALLELES, "train", "nonbinders")
per_allele_per_kfold_per_length_non_binders_test = read_data(DATA_PATH, ALLELES, "test", "nonbinders")

ALLELES

['HLA-A*03:01', 'HLA-A*01:01', 'HLA-B*27:05', 'HLA-A*02:01', 'HLA-B*07:02']

## 2.0 Merge alleles together

In [4]:
# combination_size = len(ALLELES)
ALLELES = ['HLA-A*01:01', 'HLA-A*03:01']
combination_size = 2
TOTAL_SPLITS = 5

In [5]:
import itertools
per_allele_per_kfold_per_length_binders_train_new = dict()
per_allele_per_kfold_per_length_binders_test_new = dict()
NEW_ALELLES = list()
selected_lengths = np.arange(8,20)
for combination in itertools.combinations(ALLELES, combination_size):
    allele_name = "+".join(combination)
    NEW_ALELLES.append(allele_name)
    print(allele_name)
    per_allele_per_kfold_per_length_binders_train_new[allele_name] = dict()
    per_allele_per_kfold_per_length_binders_test_new[allele_name] = dict()

    for split_num in range(TOTAL_SPLITS):
        per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num] = dict()
        per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num] = dict()

        list_of_length_sets = list()
        for old_allele in combination:
            lengths = set(per_allele_per_kfold_per_length_binders_train[old_allele][split_num].keys())
            list_of_length_sets.append(lengths)
        intersected_lengths = set.intersection(*list_of_length_sets)

        for target_length in intersected_lengths:
            data_arrays_train = list()
            data_arrays_test = list()
            if split_num == 0:
                print(f"Length {target_length}:" , end=" ")
            for old_allele in combination:
                single_allele_data_array_train = per_allele_per_kfold_per_length_binders_train[old_allele][split_num][target_length]
                if split_num == 0:
                    print(f"{old_allele} - {len(single_allele_data_array_train)}", end= " ")
                data_arrays_train.append(single_allele_data_array_train)

                single_allele_data_array_test = per_allele_per_kfold_per_length_binders_test[old_allele][split_num][target_length]
                if split_num == 0:
                    # print(f"{old_allele} - {len(single_allele_data_array_test)}", end= " ")
                    pass
                data_arrays_test.append(single_allele_data_array_test)
            print()

            per_allele_per_kfold_per_length_binders_train_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_train)
            per_allele_per_kfold_per_length_binders_test_new[allele_name][split_num][target_length] = np.concatenate(data_arrays_test)




per_allele_per_kfold_per_length_binders_train = per_allele_per_kfold_per_length_binders_train_new
per_allele_per_kfold_per_length_binders_test = per_allele_per_kfold_per_length_binders_test_new
ALLELES = NEW_ALELLES

HLA-A*01:01+HLA-A*03:01
Length 8: HLA-A*01:01 - 764 HLA-A*03:01 - 93 
Length 9: HLA-A*01:01 - 5052 HLA-A*03:01 - 5132 
Length 10: HLA-A*01:01 - 2464 HLA-A*03:01 - 2184 
Length 11: HLA-A*01:01 - 1430 HLA-A*03:01 - 691 
Length 12: HLA-A*01:01 - 1106 HLA-A*03:01 - 202 
Length 13: HLA-A*01:01 - 842 HLA-A*03:01 - 98 
Length 14: HLA-A*01:01 - 568 HLA-A*03:01 - 31 
Length 15: HLA-A*01:01 - 464 HLA-A*03:01 - 18 
Length 16: HLA-A*01:01 - 345 HLA-A*03:01 - 12 
Length 17: HLA-A*01:01 - 209 HLA-A*03:01 - 5 
Length 18: HLA-A*01:01 - 128 HLA-A*03:01 - 4 














































# 2. Train HMMS
## 2.1 Code for HMM
### 2.1.1 Training code

In [6]:
from hmm_logic_methods import train_model, save_model, add_more_states_and_reset_transitions, init_grouped_structure_model

TARGET_LENGTHS= [9]
NUM_RUNS = 2
TOTAL_SPLITS = 5

## 2.2 Train
### 2.2.1 Define treshold

In [7]:
stop_threshold = 0
cutoff = 0.0001
per_symbol_portion = -27 / 9
for target_length in TARGET_LENGTHS:
    target_length_for_training = target_length
    train_size = len(per_allele_per_kfold_per_length_binders_train[ALLELES[0]][0][target_length])
    ## Stop threshold Hardcoded here
    stop_threshold +=  abs(cutoff * train_size * per_symbol_portion  * target_length_for_training)
stop_threshold = stop_threshold / len(TARGET_LENGTHS)
stop_threshold

27.4968

### 2.2.2 Prepare models

In [22]:
# Training params
algorithm = 'baum-welch'

params = dict()


params['st'] = 25
params['multiple_check_input'] = False
params['alg'] = algorithm
params['min_iters'] = 5
params['stop_threshold'] = stop_threshold
params['emission_pseudocount'] = 1
params['transition_pseudocount'] = 0  #!CHANGE for bwv!
params['verbose'] = True
params['lr_decay'] = 0
params['edge_inertia'] = 0.25
params['distribution_inertia'] = 0

params['groups'] = 5

assert params['st'] % params['groups'] == 0

params_for_string = {key: value for key, value in params.items() if key in ['st', 'alg', 'min_iters']}
params_string = "-".join([f"{value}-{key}" for key, value in params_for_string.items()])
lengths_string = "-".join(map(str, TARGET_LENGTHS))


In [23]:
model = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

grouped_model : grouped_model-start summed to 5.0, normalized to 1.0
grouped_model : s000 summed to 17.0, normalized to 1.0
grouped_model : s001 summed to 17.0, normalized to 1.0
grouped_model : s002 summed to 17.0, normalized to 1.0
grouped_model : s003 summed to 17.0, normalized to 1.0
grouped_model : s004 summed to 17.0, normalized to 1.0
grouped_model : s005 summed to 17.0, normalized to 1.0
grouped_model : s006 summed to 17.0, normalized to 1.0
grouped_model : s007 summed to 17.0, normalized to 1.0
grouped_model : s008 summed to 17.0, normalized to 1.0
grouped_model : s009 summed to 17.0, normalized to 1.0
grouped_model : s010 summed to 17.0, normalized to 1.0
grouped_model : s011 summed to 17.0, normalized to 1.0
grouped_model : s012 summed to 17.0, normalized to 1.0
grouped_model : s013 summed to 17.0, normalized to 1.0
grouped_model : s014 summed to 17.0, normalized to 1.0
grouped_model : s015 summed to 17.0, normalized to 1.0
grouped_model : s016 summed to 17.0, normalized to 

In [24]:
model.dense_transition_matrix()

array([[0.        , 0.17647059, 0.17647059, 0.17647059, 0.17647059,
        0.05882353, 0.05882353, 0.05882353, 0.05882353, 0.05882353,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.17647059, 0.        , 0.17647059, 0.17647059, 0.17647059,
        0.05882353, 0.05882353, 0.05882353, 0.05882353, 0.05882353,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        ],
       [0.17647059, 0.17647059, 0.        , 0.17647059, 0.17647059,
        0.05882353, 0.05882353, 0.05882353, 0.05882353, 0.05882353,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.    

In [25]:
per_allele_per_run_per_split_prepared_models = dict()
for allele_name in ALLELES:
    target_allele_name = allele_name.replace('-', '_').replace('*', '_').replace(':', '_')
    per_allele_per_run_per_split_prepared_models[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = dict()
        for split_num in range(TOTAL_SPLITS):
            prepared_model  = init_grouped_structure_model(states_per_group=int(params['st']/params['groups']), number_of_groups=params['groups'])

            prepared_model.name = f'{run_num}_run-{lengths_string}-{params_string}_model-{target_allele_name}-{split_num}'
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

grouped_model : grouped_model-start summed to 5.0, normalized to 1.0
grouped_model : s000 summed to 17.0, normalized to 1.0
grouped_model : s001 summed to 17.0, normalized to 1.0
grouped_model : s002 summed to 17.0, normalized to 1.0
grouped_model : s003 summed to 17.0, normalized to 1.0
grouped_model : s004 summed to 17.0, normalized to 1.0
grouped_model : s005 summed to 17.0, normalized to 1.0
grouped_model : s006 summed to 17.0, normalized to 1.0
grouped_model : s007 summed to 17.0, normalized to 1.0
grouped_model : s008 summed to 17.0, normalized to 1.0
grouped_model : s009 summed to 17.0, normalized to 1.0
grouped_model : s010 summed to 17.0, normalized to 1.0
grouped_model : s011 summed to 17.0, normalized to 1.0
grouped_model : s012 summed to 17.0, normalized to 1.0
grouped_model : s013 summed to 17.0, normalized to 1.0
grouped_model : s014 summed to 17.0, normalized to 1.0
grouped_model : s015 summed to 17.0, normalized to 1.0
grouped_model : s016 summed to 17.0, normalized to 

In [26]:
%%time


per_name_models = dict()
per_allele_per_run_per_split_new_histories = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm=params['alg'],
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount=params['transition_pseudocount'],
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia=params['distribution_inertia'],
                                        lr_decay=params['lr_decay'])
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)


split 0
train size 10184
Shuffling for fairness 
Simple training: Train Length 10184, init_batch_size None
[1] Improvement: -173403.49986066855	Time (s): 3.013
[2] Improvement: 3243.2173410648247	Time (s): 2.335
[3] Improvement: 1467.8098758669803	Time (s): 3.52
[4] Improvement: 1010.0988978606765	Time (s): 2.803
[5] Improvement: 783.9813213470043	Time (s): 2.653
[6] Improvement: 723.1099021023838	Time (s): 3.541
[7] Improvement: 773.6655752686202	Time (s): 3.067
[8] Improvement: 855.2154059243621	Time (s): 2.568
[9] Improvement: 899.8449211426196	Time (s): 3.504
[10] Improvement: 887.1719624013058	Time (s): 4.324
[11] Improvement: 839.0024824895081	Time (s): 2.647
[12] Improvement: 780.7107752466691	Time (s): 2.208
[13] Improvement: 721.3771636474121	Time (s): 2.369
[14] Improvement: 660.423391317192	Time (s): 2.378
[15] Improvement: 600.4396890820644	Time (s): 2.208
[16] Improvement: 551.2846324378625	Time (s): 2.278
[17] Improvement: 523.3132823901542	Time (s): 2.297
[18] Improvemen

In [27]:
from hmm_visualization_methods import *

In [28]:
%%time

for i, model in enumerate(per_name_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 347 ms


In [29]:
%%time

for i, model in enumerate(per_name_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 31.7 s


In [30]:
from hmm_visualization_methods import *
per_name_pyviz = dict()
for name, model in per_name_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-25-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0) (Delete zero edges)
Converting graph for model (1_run-9-25-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0) (Delete zero edges)


## Aux viterbi

In [31]:
per_allele_per_run_per_split_new_histories = dict()

for allele_name in ALLELES:
    for run_num in range(NUM_RUNS):
        for split_num, model in per_allele_per_run_per_split_prepared_models[allele_name][run_num].items():
            prepared_model = model.copy()
            prepared_model.name = prepared_model.name + "_plus_viterbi"

            for state in prepared_model.states:
                if state is not prepared_model.start and state is not prepared_model.end:
                    state.distribution.thaw()
            per_allele_per_run_per_split_prepared_models[allele_name][run_num][split_num] = prepared_model

In [32]:
per_name_new_models = dict()
for allele_name in ALLELES:
    per_allele_per_run_per_split_new_histories[allele_name] = dict()
    for run_num in range(NUM_RUNS):
        per_kfold_per_length_data = per_allele_per_kfold_per_length_binders_train[allele_name]
        models, histories = train_model(allele_name,
                                        per_kfold_per_length_data,
                                        TARGET_LENGTHS,
                                        multiple_check_input=params['multiple_check_input'],
                                        algorithm='viterbi',
                                        verbose=params['verbose'],
                                        per_kfold_per_length_test_data=per_allele_per_kfold_per_length_binders_test[allele_name],
                                        stop_threshold=params['stop_threshold'],
                                        min_iterations= params['min_iters'],
                                        prepared_models=per_allele_per_run_per_split_prepared_models[allele_name][run_num],
                                        emission_pseudocount=params['emission_pseudocount'],
                                        transition_pseudocount=1,  #!CHANGE for bwv!
                                        edge_inertia=params['edge_inertia'],
                                        distribution_inertia= params['distribution_inertia'],
                                        lr_decay=params['lr_decay']
                                        )
        per_allele_per_run_per_split_prepared_models[allele_name][run_num] = models
        per_allele_per_run_per_split_new_histories[allele_name][run_num] = histories
        # per_allele_per_run_per_split_models[allele_name][run_num] = models
        for split_num, model in models.items():
            per_name_new_models[model.name] = model
        # save models
        for split_num, model in models.items():
            history = histories[split_num]
            save_model(TARGET_PATH_TO_RESULTS, model,history)

split 0
train size 10184
Shuffling for fairness 
Simple training: Train Length 10184, init_batch_size None
[1] Improvement: -1821.6599836245005	Time (s): 1.207
[2] Improvement: 604.7921997361118	Time (s): 1.238
[3] Improvement: 180.45742717685062	Time (s): 1.162
[4] Improvement: 57.89363833528478	Time (s): 1.159
[5] Improvement: 45.02678113649017	Time (s): 1.206
[6] Improvement: 81.4353144504712	Time (s): 1.208
[7] Improvement: 127.5166325836326	Time (s): 1.211
[8] Improvement: 44.9840312206361	Time (s): 1.223
[9] Improvement: 27.28186049099895	Time (s): 1.161
Total Training Improvement: -652.2720984940242
Total Training Time (s): 11.9580
split 0
train size 10184
Shuffling for fairness 
Simple training: Train Length 10184, init_batch_size None
[1] Improvement: -1411.213628217869	Time (s): 1.213
[2] Improvement: 546.447453929286	Time (s): 1.211
[3] Improvement: 71.91856549523072	Time (s): 1.206
[4] Improvement: -9.080759238655446	Time (s): 1.23
[5] Improvement: 11.302177349600242	Time (

In [33]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    print(i)
    # with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph.png", 'w+') as f:
    #     model.plot(file=f, crop_zero=False)
    with open(f"{TARGET_PATH_TO_RESULTS}{model.name}/state_graph_cropped.png", 'w+') as f:
        model.plot(file=f, crop_zero=True)

0
1
Wall time: 1.64 s


In [34]:
%%time

for i, model in enumerate(per_name_new_models.values()):
    plot_distributions_for_states(model,TARGET_PATH_TO_RESULTS, horizontal=False)
    print(i)

0
1
Wall time: 31.8 s


In [35]:
per_name_pyviz = dict()
for name, model in per_name_new_models.items():
    per_name_pyviz[name] = make_pyviz_graph(model, TARGET_PATH_TO_RESULTS, precision=3)

Converting graph for model (0_run-9-25-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0_plus_viterbi) (Delete zero edges)
Converting graph for model (1_run-9-25-st-baum-welch-alg-5-min_iters_model-HLA_A_01_01+HLA_A_03_01-0_plus_viterbi) (Delete zero edges)
